<div style="background:linear-gradient(135deg,#0a2540 0%,#1a3a5c 60%,#0f3460 100%);
            padding:40px 30px;border-radius:12px;text-align:center;margin-bottom:10px;">
  <h1 style="color:#f4a261;font-size:2em;margin:0 0 8px;">
    🔬 Ciencia de Datos en Descubrimiento de Fármacos
  </h1>
  <h2 style="color:#a8dadc;font-size:1.2em;font-weight:400;margin:0 0 16px;">
    04 · Quimioinformatica y RDKit
  </h2>
  <p style="color:#cdd6f4;font-size:0.95em;max-width:620px;margin:0 auto;line-height:1.6;">
    Universidad Nacional de Colombia · Extensión UNAL 2026<br>
    <em>Semana 2 — Sin experiencia previa en programación</em>
  </p>
</div>


# Introducción a la quimioinformática usando RDKit

---
### En esta lección aprenderás:

- cómo leer SMILES con `rdkit`.
- cómo manipular y visualizar moléculas.
- cómo calcular descriptores moleculares.
- cómo calcular la similitud entre moléculas usando fingerprints.

---

El notebook de hoy es parte de una serie de notebooks de quimioinformática que acompañan el curso **Ciencia de Datos en Descubrimiento de Fármacos**.
El fármaco de ejemplo es el **sorafenib**, un inhibidor de quinasas aprobado para el tratamiento de varios tipos de cáncer (carcinoma hepatocelular, carcinoma de células renales, cáncer de tiroides diferenciado).

**Target:** RAF quinasa y receptores de factores de crecimiento (VEGFR, PDGFR)

---

In [ ]:
sorafenib = "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(c3)C(F)(F)F)cc2)ccn1" # Escribe el SMILES en las comillas
print(sorafenib)
type(sorafenib)

Como puedes ver, los SMILES se almacenan como `str` (`string`). En realidad podemos manipular este `string` y también aplicarle funciones. Sin embargo, el problema es que aunque Python entiende los SMILES como un `string`, no puede inferir la estructura molecular subyacente a partir de ellos. No tenemos forma de obtener información sobre el peso molecular, la carga, la aromaticidad, etc. solo a partir del `string`.

Para este fin se necesita `rdkit`, una librería de quimioinformática de código abierto para Python:

In [ ]:
# Installs RDKit
!pip install rdkit

In [ ]:
from rdkit.Chem import AllChem as Chem
from rdkit.Chem.Draw import IPythonConsole

sorafenib = Chem.MolFromSmiles(sorafenib)
sorafenib

Con la ayuda de RDKit, los SMILES pueden leerse y representarse como una molécula válida.
El `type(sorafenib)` ahora es:

In [ ]:
type(sorafenib)

La función `Chem.MolFromSmiles` convierte el string SMILES en un nuevo tipo de variable: el **RDKit-Mol**. Mientras una molécula esté almacenada como `rdkit.Chem.rdchem.Mol` en Python, puedes aplicarle todas las funciones de rdkit. También puedes usar `Chem.MolToSmiles(mol)` para obtener la molécula en formato SMILES de nuevo.

Observa lo que ocurre cuando introduces un SMILES inválido:

In [ ]:
Chem.MolToSmiles(sorafenib)

El `string` SMILES del sorafenib ahora se ve diferente al que leíste originalmente. La diferencia está en la representación de los anillos aromáticos. En el string original se usaban explícitamente dobles enlaces `=`, pero ahora las `C` mayúsculas con doble enlace son reemplazadas por `c` minúsculas. RDKit canonicaliza automáticamente los SMILES. Esto significa que independientemente del formato en que introduzcas el SMILES, siempre obtendrás la misma representación canónica. Esto es muy útil para eliminar duplicados en bases de datos.

In [ ]:
Chem.MolFromSmiles('CNC(=[O-])c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1') # (=[O-]) en vez de (=O)

### RDKit
Ahora que tienes el sorafenib en el formato correcto, también puedes obtener información sobre esta molécula:

In [ ]:
sorafenib.GetNumAtoms() # Cuanto atomos pesados tiene el Sorafenib

Existen varias funciones que se pueden usar para obtener información sobre las moléculas. `rdkit` asigna un índice a cada átomo y enlace. Con este índice puedes seleccionar átomos o enlaces individuales. Puedes ver qué índice tiene cada átomo cambiando las opciones de `Draw` de la siguiente manera:

In [ ]:
from rdkit.Chem import Draw 
IPythonConsole.drawOptions.addAtomIndices = True 
IPythonConsole.drawOptions.addBondIndices = False
IPythonConsole.molSize = (500, 500) 

In [ ]:
sorafenib

Los átomos individuales pueden seleccionarse por su índice con `.GetAtomWithIdx()`. Otras funciones permiten obtener más información sobre cada átomo:

In [ ]:
print("Simbolo del atomo con index 3")
print(sorafenib.GetAtomWithIdx(3).GetSymbol())

print("\nMasa del atomo index 3")
print(sorafenib.GetAtomWithIdx(3).GetMass())

print("\nHibridización del atomo con indice 3")
print(sorafenib.GetAtomWithIdx(3).GetHybridization())


Con la función `.SetAtomicNum()` también puedes cambiar átomos individuales y, por ejemplo, convertir la cetona en una imina.

In [ ]:
sorafenib.GetAtomWithIdx(3).SetAtomicNum(7)
display(sorafenib)
print(Chem.MolToSmiles(sorafenib))
sorafenib.GetAtomWithIdx(3).SetAtomicNum(8) # Cambio reversado de nuevo

¿Puedes reemplazar uno de los átomos de flúor por un átomo de carbono?

In [ ]:
sorafenib._____.______ # Escribe tu solucion aqui

display(sorafenib)
print(Chem.MolToSmiles(sorafenib))
sorafenib = Chem.MolFromSmiles("CNC(=O)C1=NC=CC(=C1)OC2=CC=C(C=C2)NC(=O)NC3=CC(=C(C=C3)Cl)C(F)(F)F")

<details>
<summary><strong>Solución:</strong></summary>

```python
    sorafenib.GetAtomWithIdx(31).SetAtomicNum(6)
```
</details>

También se pueden usar funciones similares para los enlaces. A cada enlace también se le asigna un índice.

In [ ]:
IPythonConsole.drawOptions.addAtomIndices = False # No muestra los indices de los atomos
IPythonConsole.drawOptions.addBondIndices = True # Muestra los indices de los enlaces

sorafenib

In [ ]:
print("Que tipo de enlace es el enlace 4")
print(sorafenib.GetBondWithIdx(4).GetBondType())

print("\nEl enlace 4 esta en un anillo de 7 miembros")
print(sorafenib.GetBondWithIdx(4).IsInRingSize(7))

print("\nEl enlace 4 esta en un anillo de 6 miembros")
print(sorafenib.GetBondWithIdx(4).IsInRingSize(6))

IPythonConsole.drawOptions.addBondIndices = False

---
## SMARTS — Búsqueda por Subestructura

Hasta ahora hemos trabajado con SMILES, que describe una molécula específica.
**SMARTS** (*SMILES Arbitrary Target Specification*) es una extensión de SMILES que permite definir **patrones de subestructura**: en lugar de buscar una molécula exacta, buscamos un fragmento o patrón que puede estar presente en muchas moléculas.

### ¿Para qué sirve SMARTS en drug discovery?
- Buscar si una molécula contiene un grupo funcional específico
- Filtrar compuestos problemáticos (PAINS, BRENKs)
- Definir farmacoforores
- Marcar sitios reactivos

### Diferencias clave entre SMILES y SMARTS

| | SMILES | SMARTS |
|--|--------|--------|
| Propósito | Describir **una molécula** | Describir un **patrón** |
| `C` | Carbono alifático | Cualquier carbono alifático |
| `c` | Carbono aromático | Cualquier carbono aromático |
| `[#6]` | No válido | Cualquier carbono (alifático o aromático) |
| `[!N]` | No válido | Cualquier átomo excepto N |
| `,` | No válido | OR lógico: `[N,O]` = N ó O |

> 💡 **En RDKit:** `Chem.MolFromSmarts(patron)` convierte un SMARTS en un objeto de patrón,
> y `mol.HasSubstructMatch(patron)` devuelve `True` si la molécula contiene ese patrón.


In [ ]:
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

# Molécula de trabajo: sorafenib
sorafenib = Chem.MolFromSmiles("CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1")
print("✅ Sorafenib cargado")
print(f"   Átomos: {sorafenib.GetNumAtoms()}")
print(f"   SMILES canónico: {Chem.MolToSmiles(sorafenib)[:50]}...")


### 1. Búsqueda básica: ¿tiene sorafenib un grupo urea?

El sorafenib es conocido por tener un grupo **urea** (`-NH-C(=O)-NH-`), que es fundamental
para su unión a RAF quinasa. Definimos el patrón SMARTS y buscamos si está presente.


In [ ]:
# Definir el patrón SMARTS del grupo urea: N-C(=O)-N
patron_urea = Chem.MolFromSmarts("[NH]-C(=O)-[NH]")

# Buscar si sorafenib contiene el patrón
tiene_urea = sorafenib.HasSubstructMatch(patron_urea)
print(f"¿Sorafenib tiene grupo urea? {tiene_urea}")

# Obtener los índices de los átomos que forman el match
matches = sorafenib.GetSubstructMatches(patron_urea)
print(f"Átomos que forman el patrón: {matches}")

# Visualizar el match resaltado en la molécula
Draw.MolToImage(sorafenib,
                highlightAtoms=list(matches[0]),
                size=(400, 300))


### 2. Identificar grupos funcionales en una molécula

Los SMARTS son la forma estándar de detectar grupos funcionales automáticamente.
A continuación definimos un diccionario con patrones comunes en química medicinal
y lo aplicamos al sorafenib.


In [ ]:
# Diccionario de grupos funcionales con sus SMARTS
grupos_funcionales = {
    "Urea":              "[NH]-C(=O)-[NH]",
    "Amida":             "[NX3][CX3](=[OX1])",
    "Amina aromática":   "[NH2]c",
    "Anillo aromático":  "c1ccccc1",
    "Anillo piridina":   "n1ccccc1",
    "Halógeno":          "[F,Cl,Br,I]",
    "Éter":              "[OX2]([#6])[#6]",
    "CF3":               "[CX4](F)(F)F",
}

print(f"Grupos funcionales en Sorafenib:")
print("=" * 45)
for nombre, smarts in grupos_funcionales.items():
    patron = Chem.MolFromSmarts(smarts)
    if patron is None:
        print(f"  ⚠️  {nombre:<22}: SMARTS inválido")
        continue
    n_matches = len(sorafenib.GetSubstructMatches(patron))
    presente = "✅" if n_matches > 0 else "❌"
    print(f"  {presente} {nombre:<22}: {n_matches} ocurrencia(s)")


### 3. Ejercicio: detectar grupos funcionales en múltiples moléculas

Ahora aplica el mismo análisis a las alternativas del sorafenib.
¿Puedes escribir un loop que detecte si cada molécula tiene un grupo **amida** y un **halógeno**?


In [ ]:
smiles_alternativas = [
    "CNC(=O)c1cc(Oc2ccc(NC(=S)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "N#Cc1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc2)ccn1",
]
mols_alt = [Chem.MolFromSmiles(s) for s in smiles_alternativas]

patron_amida   = Chem.MolFromSmarts("______")  # ← escribe el SMARTS de amida
patron_halogeno = Chem.MolFromSmarts("______") # ← escribe el SMARTS de halógeno

print(f"{'Mol':>4}  {'Amida':>8}  {'Halógeno':>10}")
print("-" * 28)
for i, mol in enumerate(mols_alt):
    tiene_amida    = ______.HasSubstructMatch(______)  # ← completa
    tiene_halogeno = ______.HasSubstructMatch(______)  # ← completa
    print(f"  {i+1:>2}    {'✅' if tiene_amida else '❌':>5}    {'✅' if tiene_halogeno else '❌':>8}")


<details>
<summary><strong>Solución:</strong></summary>

```python
patron_amida    = Chem.MolFromSmarts("[NX3][CX3](=[OX1])")
patron_halogeno = Chem.MolFromSmarts("[F,Cl,Br,I]")

for i, mol in enumerate(mols_alt):
    tiene_amida    = mol.HasSubstructMatch(patron_amida)
    tiene_halogeno = mol.HasSubstructMatch(patron_halogeno)
    print(f"  {i+1:>2}    {'✅' if tiene_amida else '❌':>5}    {'✅' if tiene_halogeno else '❌':>8}")
```
</details>


### 4. Visualizar patrones SMARTS resaltados

RDKit permite resaltar en color los átomos que coinciden con un patrón SMARTS.
Esto es muy útil para verificar visualmente que el patrón está siendo detectado correctamente.


In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import SVG, display

def resaltar_patron(mol, smarts, titulo=""):
    """Dibuja una molécula resaltando los átomos que coinciden con el SMARTS dado."""
    patron = Chem.MolFromSmarts(smarts)
    if patron is None:
        print(f"SMARTS inválido: {smarts}")
        return
    matches = mol.GetSubstructMatches(patron)
    if not matches:
        print(f"Patrón no encontrado en la molécula")
        return

    # Aplanar todos los índices de átomos del match
    atomos_match = [idx for match in matches for idx in match]
    img = Draw.MolToImage(mol, highlightAtoms=atomos_match, size=(400, 280))
    display(img)
    print(f"  Patrón '{smarts}' encontrado en {len(matches)} lugar(es)")
    print(f"  Átomos resaltados: {atomos_match}")

print("=== Grupo urea en sorafenib ===")
resaltar_patron(sorafenib, "[NH]-C(=O)-[NH]")

print("\n=== Grupos CF3 en sorafenib ===")
resaltar_patron(sorafenib, "[CX4](F)(F)F")

print("\n=== Anillos aromáticos en sorafenib ===")
resaltar_patron(sorafenib, "c1ccccc1")


### 5. Ejercicio: SMARTS con operadores booleanos

Los SMARTS permiten usar operadores lógicos:
- `,` = **OR**: `[N,O]` coincide con N ó O
- `&` = **AND** (alta prioridad): `[C&H3]` = carbono con exactamente 3 H
- `;` = **AND** (baja prioridad): `[#6;r6]` = carbono en anillo de 6 miembros
- `!` = **NOT**: `[!N]` = cualquier átomo excepto N

**Tarea:** Escribe un SMARTS que detecte átomos de nitrógeno **que no sean** parte de un anillo aromático.
Luego, encuentra cuántos hay en el sorafenib.


In [ ]:
# Pista: un átomo aromático en SMARTS se representa con la letra minúscula 'a'
# Un átomo no aromático: A (mayúscula)
# Para "nitrógeno no aromático": nitrógeno AND NOT aromático

patron_N_no_arom = Chem.MolFromSmarts("______")  # ← escribe aquí el SMARTS

if patron_N_no_arom:
    matches = sorafenib.GetSubstructMatches(patron_N_no_arom)
    print(f"Nitrógenos no aromáticos en sorafenib: {len(matches)}")
    print(f"En índices de átomos: {[m[0] for m in matches]}")
    resaltar_patron(sorafenib, Chem.MolToSmarts(patron_N_no_arom))


<details>
<summary><strong>Solución:</strong></summary>

```python
# [N] = nitrógeno  ;  a = átomo aromático  ;  ! = NOT
# Nitrógeno que NO sea aromático:
patron_N_no_arom = Chem.MolFromSmarts("[N;!a]")

# Alternativa equivalente:
patron_N_no_arom = Chem.MolFromSmarts("[NX3;!$(nc)]")
```
El sorafenib tiene 2 nitrógenos no aromáticos: los dos NH de la urea y la amida.
</details>


### 6. Ejercicio: filtrar una lista de moléculas por SMARTS

Una aplicación clave de SMARTS es filtrar datasets para **quedarse solo con moléculas
que contienen un fragmento específico** — por ejemplo, todos los inhibidores de quinasa
que tienen un grupo urea o amida como ancla de unión al hinge de la proteína.

**Tarea:** Dada la lista `smiles_quinolonas`, conserva solo las moléculas que contengan
un anillo de **piridina** (patrón: `n1ccccc1`) O una **amina terciaria alifática** (`[NX3;H0;!$(N=*)]`).


In [ ]:
smiles_quinolonas = [
    "CCN1C=C(C(=O)c2cc(Oc3ccc(NC(=O)Nc4ccc(Cl)c(C(F)(F)F)c4)cc3)ccn2)C(=O)O",  # sorafenib-like
    "C1CN(CCN1)c2c(F)cc3c(=O)c(cn3CC)C(=O)O",   # quinolona con piperazina
    "O=C(O)c1cn(CC)c2cc(F)c(N3CCNCC3)cc12",     # norfloxacino-like
    "CC1CCN(CC1)c2c(F)cc3c(=O)c(cn3C4CC4)C(=O)O",  # ciprofloxacino-like
    "c1ccc(NC(=O)c2ccccc2)cc1",                 # simple anilida
]
mols_q = [Chem.MolFromSmiles(s) for s in smiles_quinolonas if Chem.MolFromSmiles(s)]

patron_piridina      = Chem.MolFromSmarts("n1ccccc1")
patron_amina_tert    = Chem.MolFromSmarts("[NX3;H0;!$(N=*)]")

# Filtra: moléculas que tienen piridina O amina terciaria
mols_filtradas = [mol for mol in mols_q
                  if ___________________  # ← completa la condición
                  ]

print(f"Moléculas totales:   {len(mols_q)}")
print(f"Moléculas filtradas: {len(mols_filtradas)}")
Draw.MolsToGridImage(mols_filtradas, subImgSize=(280, 220))


<details>
<summary><strong>Solución:</strong></summary>

```python
mols_filtradas = [mol for mol in mols_q
                  if mol.HasSubstructMatch(patron_piridina)
                  or mol.HasSubstructMatch(patron_amina_tert)]
```
</details>


---
## Filtros de calidad: PAINS y BRENKs

En el descubrimiento de fármacos, muchos compuestos dan **falsos positivos** en ensayos
de high-throughput screening (HTS). Dos filtros basados en SMARTS son los más usados:

### PAINS (Pan-Assay Interference Compounds)
Publicados por **Baell & Holloway (J. Med. Chem. 2010)**: 480 subestructuras SMARTS
que identifican compuestos que interfieren de forma no específica por:
- **Agregación** — forman micelas que atrapan proteínas
- **Redox cycling** — genera ROS artefactuales
- **Reactividad** — alquilación inespecífica de nucleófilos
- **Fluorescencia** — absorben en el rango del ensayo

### BRENKs (Brenk filters)
Publicados por **Brenk et al. (ChemMedChem 2008)**: 105 SMARTS para grupos funcionales
con toxicidad potencial, inestabilidad química o propiedades ADME deficientes.

> ⚠️ **Importante:** Estos filtros **marcan** compuestos, no los eliminan automáticamente.
> Un compuesto marcado puede ser activo genuinamente — requiere inspección manual.


### 7. Filtro PAINS con RDKit FilterCatalog

RDKit incluye los 480 patrones PAINS listos para usar en `FilterCatalog`.
Esta es la forma más robusta de aplicarlos — sin tener que definir los SMARTS manualmente.


In [ ]:
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams

# Construir el catálogo PAINS
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
catalogo_pains = FilterCatalog.FilterCatalog(params)

# Moléculas de prueba — algunas son PAINS conocidos
moleculas_prueba = {
    "Sorafenib":          "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "Rhodanina (PAINS)":  "O=C1CSC(=S)N1",          # scaffold PAINS clásico
    "Catecol (PAINS)":    "Oc1ccccc1O",              # quelante, da falsos positivos
    "Curcumina (PAINS)":  "COc1cc(/C=C/C(=O)CC(=O)/C=C/c2ccc(O)c(OC)c2)ccc1O",
    "Aspirina":           "CC(=O)Oc1ccccc1C(=O)O",   # no PAINS
    "Ibuprofeno":         "CC(C)Cc1ccc(cc1)C(C)C(=O)O",  # no PAINS
}

print(f"{'Molécula':<25} {'PAINS':>8}  {'Categoría'}")
print("=" * 65)
for nombre, smiles in moleculas_prueba.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    entrada = catalogo_pains.GetFirstMatch(mol)
    es_pains = entrada is not None
    categoria = entrada.GetDescription() if entrada else "—"
    icono = "🔴 SÍ" if es_pains else "✅ NO"
    print(f"  {nombre:<23} {icono:>9}  {categoria}")


### 8. Filtro BRENKs con SMARTS manuales

Los filtros de Brenk no están en RDKit FilterCatalog por defecto, pero podemos
implementarlos directamente con sus SMARTS. A continuación definimos los más importantes:


In [ ]:
# ── Subconjunto representativo de filtros BRENKs ──────────────────────────
BRENKS_SMARTS = {
    # ── Grupos reactivos / tóxicos ──────────────────────────────────────
    "Epóxido (genotóxico)":          "[OX2r3]1[#6r3][#6r3]1",
    "Aldehído (reactivo)":           "[CX3H1](=O)[#6]",
    "Nitro aromático (mutagénico)":  "[$([$([NX3](=O)=O)]),$([NX3+](=O)[O-])]c",
    "Isocianato (reactivo)":         "[NX2](=O)=O",
    "Azida (explosivo)":             "[N-]=[N+]=[N,C]",
    "Cloruro de ácido":              "ClC(=O)",
    "Sulfonato (electrófilo)":       "OS(=O)(=O)[Cl,F]",

    # ── Inestabilidad química ─────────────────────────────────────────
    "Diazo":                         "[$([#6]=[N+]=[N-]),$([#6-]-[N+]#N)]",
    "Hemiacetal":                    "[OX2H][CX4][OX2]",
    "Acetal":                        "[OX2][CX4][OX2]",

    # ── ADME problemático ─────────────────────────────────────────────
    "Ácido bórónico (hidrolizable)": "[B]([OH])[OH]",
    "Tiol (mala ADME)":              "[SX2H]",
    "Trihalometilo":                 "[CX4]([F,Cl,Br])([F,Cl,Br])[F,Cl,Br]",
}

# Compilar todos los patrones
patrones_brenks = {}
for nombre, smarts in BRENKS_SMARTS.items():
    patron = Chem.MolFromSmarts(smarts)
    if patron is not None:
        patrones_brenks[nombre] = patron
    else:
        print(f"⚠️  SMARTS inválido: {nombre}")

print(f"✅ {len(patrones_brenks)} filtros BRENKs cargados correctamente")


In [ ]:
def aplicar_brenks(mol):
    """
    Aplica los filtros BRENKs a una molécula.
    Retorna una lista de los filtros que activan (lista vacía = limpia).
    """
    alertas = []
    for nombre, patron in patrones_brenks.items():
        if mol.HasSubstructMatch(patron):
            alertas.append(nombre)
    return alertas

# Probar con moléculas de ejemplo
moleculas_brenk = {
    "Sorafenib":              "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "Epóxido simple":         "C1CO1",
    "Nitrobenceno":           "O=[N+]([O-])c1ccccc1",
    "Benzaldehído":           "O=Cc1ccccc1",
    "Ácido fenilbórónico":    "OB(O)c1ccccc1",
    "Tricloro-metano (CHCl3)":"ClC(Cl)Cl",
    "Aspirina":               "CC(=O)Oc1ccccc1C(=O)O",
}

print("ANÁLISIS DE FILTROS BRENKS")
print("=" * 65)
for nombre, smiles in moleculas_brenk.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    alertas = aplicar_brenks(mol)
    if alertas:
        print(f"  🔴 {nombre}")
        for alerta in alertas:
            print(f"       └─ {alerta}")
    else:
        print(f"  ✅ {nombre}  — sin alertas BRENKs")


### 9. Pipeline integrado: PAINS + BRENKs sobre el dataset del curso

Ahora aplicamos ambos filtros de forma integrada al dataset de alternativas al sorafenib.
Este es el tipo de análisis que harías en la **semana 3 (curación de datos)** del curso.


In [ ]:
# Dataset completo: sorafenib + alternativas
dataset = {
    "Sorafenib":    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "Alternativa 1":"CNC(=O)c1cc(Oc2ccc(NC(=S)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "Alternativa 2":"C[C@@H](NC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1)C(=O)NO",
    "Alternativa 3":"CNC(=O)c1cc(Oc2ccc(NC(=S)Nc3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc2)ccn1",
    "Rhodanina":    "O=C1CSC(=S)N1",          # PAINS clásico
    "Nitrobenceno": "O=[N+]([O-])c1ccccc1",   # BRENK
    "Aspirina":     "CC(=O)Oc1ccccc1C(=O)O",
}

print(f"{'Molécula':<18} {'PAINS':>8} {'BRENKs':>8}  Estado")
print("=" * 65)
resultados = []
for nombre, smiles in dataset.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue

    # PAINS
    entrada_pains = catalogo_pains.GetFirstMatch(mol)
    es_pains = entrada_pains is not None
    cat_pains = entrada_pains.GetDescription()[:20] if entrada_pains else "—"

    # BRENKs
    alertas_brenks = aplicar_brenks(mol)
    n_brenks = len(alertas_brenks)

    # Estado final
    if es_pains or n_brenks > 0:
        estado = "⚠️  MARCAR"
    else:
        estado = "✅ OK"

    resultados.append({
        "nombre": nombre,
        "mol": mol,
        "pains": es_pains,
        "brenks": n_brenks,
        "ok": not es_pains and n_brenks == 0
    })
    print(f"  {nombre:<17} {'🔴' if es_pains else '—':>6}  {n_brenks:>5}    {estado}")

print()
n_ok = sum(1 for r in resultados if r["ok"])
print(f"Resumen: {n_ok}/{len(resultados)} moléculas pasan ambos filtros")


In [ ]:
# Visualizar solo las moléculas que pasan los filtros
mols_ok   = [r["mol"]    for r in resultados if r["ok"]]
names_ok  = [r["nombre"] for r in resultados if r["ok"]]

print(f"Moléculas que pasan PAINS + BRENKs: {len(mols_ok)}")
Draw.MolsToGridImage(mols_ok,
                     legends=names_ok,
                     subImgSize=(300, 250),
                     molsPerRow=3)


In [ ]:
# Visualizar las moléculas marcadas con la razón
mols_marcadas  = [r["mol"]    for r in resultados if not r["ok"]]
names_marcadas = [r["nombre"] for r in resultados if not r["ok"]]

print(f"Moléculas marcadas: {len(mols_marcadas)}")
if mols_marcadas:
    Draw.MolsToGridImage(mols_marcadas,
                         legends=names_marcadas,
                         subImgSize=(300, 250),
                         molsPerRow=3)


### 💡 Resumen de la sección SMARTS

| Concepto | Función RDKit | Cuándo usar |
|----------|--------------|-------------|
| Definir patrón | `Chem.MolFromSmarts(smarts)` | Siempre que vayas a buscar |
| Detectar presencia | `mol.HasSubstructMatch(patron)` | Filtrado binario |
| Obtener índices | `mol.GetSubstructMatches(patron)` | Visualización y análisis |
| PAINS automático | `FilterCatalog.FilterCatalog(params)` | Curación de datasets |
| BRENKs manuales | Diccionario de SMARTS + `HasSubstructMatch` | Grupos funcionales problemáticos |

En el **NB-DATA-02** (Semana 3) aplicarás estos filtros a datasets completos de ChEMBL
para la curación de datos antes del modelado.

---


### Descriptores

Más útil que la información sobre átomos individuales son los descriptores calculados para una molécula completa. Con diferentes submódulos de `rdkit` puedes calcular distintas propiedades de las moléculas:

In [ ]:
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.Crippen import MolLogP

print("LogP",MolLogP(sorafenib))
print("Peso Molecular",MolWt(sorafenib))

## Alternativas al Sorafenib

El objetivo es encontrar moléculas alternativas al sorafenib. Ya se ha realizado una preselección. Los SMILES se encuentran en la lista `smiles`.

In [ ]:
smiles = [
    "CNC(=O)c1cc(Oc2ccc(NC(=S)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "C[C@@H](NC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1)C(=O)NO",
    "CNC(=O)c1cc(Oc2ccc(NC(=S)Nc3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc2)ccn1",
    "N#Cc1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "CN(C)c1ccc(NC(=O)c2cc(Oc3ccc(NC(=O)Nc4ccc(Cl)c(C(F)(F)F)c4)cc3)ccn2)cc1", 
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Br)c(C(F)(F)F)c3)cc2)ccn1",
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(OC(F)(F)F)cc3)cc2)ccn1",
    "CCNC(=O)c1cc(Oc2ccc(NC(=O)Nc3ccc(Cl)c(C(F)(F)F)c3)cc2)ccn1",
    "CNC(=O)c1cc(Oc2ccc(NC(=O)Nc3cccc(C(F)(F)F)c3)cc2)ccn1"
]

Para evitar tener que convertir cada SMILES individualmente en un objeto `mol`, escribe un `for loop`.

Puedes mostrar múltiples moléculas una al lado de la otra con la función `Draw.MolsToGridImage(mols)`.

In [ ]:
mols = [ _____ for x in ______] # Escribe tu solucion aqui
Draw.MolsToGridImage(mols,subImgSize=(300, 300)) #subImgSize permite mostrar imagenes mas grandes

<details>
<summary><strong>Solución:</strong></summary>

```python
mols = [Chem.MolFromSmiles(x) for x in smiles]
Draw.MolsToGridImage(mols,subImgSize=(300, 300))
```
</details>

Para evitar costos innecesarios, debes seleccionar únicamente las moléculas más prometedoras. Para esto, puedes aplicar lo que has aprendido hasta ahora.
Una regla de oro simple pero importante en el desarrollo de fármacos es la ["Regla de los Cinco de Lipinski"](https://en.wikipedia.org/wiki/Lipinski%27s_rule_of_five). Establece que las moléculas con buena biodisponibilidad oral suelen cumplir con los siguientes cuatro criterios:

- Peso molecular ≤ 500 Da
- LogP ≤ 5 (lipofilia)
- Número de donadores de puentes de hidrógeno ≤ 5
- Número de aceptores de puentes de hidrógeno ≤ 10

Una molécula que viola más de una de estas reglas puede tener problemas de biodisponibilidad oral. Sin embargo, hay excepciones notables como la ciclosporina A.

Ya calculaste el valor de LogP y el peso molecular con funciones de `rdkit`.
El submódulo `Lipinski` en RDKit ofrece más funciones, entre ellas para el cálculo del número de donadores y aceptores de puentes de hidrógeno.

Primero calcula el número de donadores de hidrógeno (`NumHDonors()`) para cada molécula en `mols`:

In [ ]:
from rdkit.Chem.Lipinski import NumHAcceptors, NumHDonors

NumDonors = [______(x) for x in ______] # Escribe tu solucion aquí

<details>
<summary><strong>Solución:</strong></summary>

```python
NumDonors = [NumHDonors(x) for x in mols]
```
</details>

Para mostrar el número de donadores junto con las moléculas puedes usar la función `MolsToGridImage()`. Aquí simplemente tienes que pasar `NumDonors` a la variable de entrada `legends`. El problema es que la función siempre espera `Strings`, no `integers`. Por eso usamos otro `for-loop` para convertir los `integers` en `strings`:

In [ ]:
NumDonors = [str(x) for x in NumDonors] # Convertir la lista de ints en strings
Draw.MolsToGridImage(mols, legends = NumDonors,subImgSize = (300,300))

En el pie de imagen puedes ver el número de donadores de puentes de hidrógeno. Todas las moléculas tienen menos donadores que el límite máximo establecido por Lipinski. Podemos repetir lo mismo para los aceptores.
Sin embargo, esta vez escribe el `for-loop` de manera que los `integers` se conviertan inmediatamente en `strings`. Así evitas tener que hacer la conversión en un paso separado:

In [ ]:
NumAcceptors = [str(________) for x in ________] # Escribe tu solución Aquí
Draw.MolsToGridImage(mols, legends = NumAcceptors, subImgSize= (300,300))

<details>
<summary><strong>Solución:</strong></summary>

```python
NumAcceptors = [str(NumHAcceptors(x)) for x in mols]
Draw.MolsToGridImage(mols, legends = NumAcceptors, subImgSize= (300,300))
```
</details>

Ninguna de las moléculas viola la regla de Lipinski tampoco para los aceptores.
Ahora calcula el peso molecular (`MolWt()`) para las alternativas al sorafenib.

In [ ]:
molWeight = [str(_________) for __ in _______] # Escribe tu solución Aquí
Draw.MolsToGridImage(mols, legends = molWeight, subImgSize=(300, 300))

<details>
<summary><strong>Solución:</strong></summary>

```python
molWeight = [str(MolWt(x)) for x in mols]
Draw.MolsToGridImage(mols, legends = molWeight, subImgSize=(300, 300))
```
</details>

Algunas moléculas son en realidad más pesadas de lo que permite la "Regla de los Cinco de Lipinski".

Lo último que harás es calcular el LogP (`MolLogP()`).

In [ ]:
logP = [_______ for ____ in _____] # Escribe tu solución Aquí, recuerda el str() en la solución
Draw.MolsToGridImage(mols, legends = logP,subImgSize=(300, 300))

<details>
<summary><strong>Solución:</strong></summary>

```python
logP = [str(MolLogP(x)) for x in mols]
Draw.MolsToGridImage(mols, legends = logP,subImgSize=(300, 300))
```
</details>

De hecho, la mayoría de las moléculas supera el valor de LogP de Lipinski. Solo tres moléculas tienen un valor menor a cinco. Estas últimas dos moléculas son las únicas que cumplen con las cuatro reglas de Lipinski. Por lo tanto, pueden ser especialmente adecuadas como fármacos. Sin embargo, no se deben descartar todas las demás moléculas solo por eso — la Regla de los Cinco de Lipinski es solo una guía y hay excepciones importantes.

Con ayuda de `numpy` podemos crear un `array` booleano que muestre qué moléculas cumplen con la condición de LogP < 5:

In [ ]:
import numpy as np

logP = [MolLogP(x) for x in mols]
logP = np.array(logP) # La lista se convierte en un array de numpy
logP < 5.0

Ahora haz lo mismo para el peso molecular (`MolWt()`).

In [ ]:
molWeight = [____() ___ ____ ___ ___] # Escribe tu solución Aquí
molWeight = ______________ # convierte la lista en un array
molWeight < 500

<details>
<summary><strong>Solución:</strong></summary>

```python
molWeight = [MolWt(x) for x in mols]
molWeight = np.array(molWeight)
molWeight < 500
```
</details>

Para seleccionar las moléculas que tienen un peso inferior a 500 **o** un LogP inferior a cinco, puedes usar el símbolo `|`. El `|` representa "o" (OR lógico). La expresión `(logP < 5) | (molWeight < 500)` dará `True` para los elementos que cumplan al menos una de las dos condiciones. Se dará `False` si un elemento no cumple ninguna condición.

In [ ]:
(logP < 5) | (molWeight<500)

Con este array de booleanos (`bool`) ahora podemos seleccionar las moléculas que pasan el filtro.

In [ ]:
mols=np.array(mols)
mols_subset=mols[(logP < 5) | (molWeight<500)] # Nosotros convertimos la lista en un array de numpy
Draw.MolsToGridImage(mols_subset, subImgSize=(300, 300))

Calculando descriptores, puedes reducir el número de moléculas candidatas. En el siguiente paso aprenderás cómo reducir aún más la selección con una búsqueda de similitud.

## Fingerprints y Búsqueda de Similitud

RDKit también puede calcular varios *fingerprints moleculares*. Entre ellos se encuentra el *Extended Connectivity Fingerprint* (ECFP), desarrollado originalmente por [Hahn et al.](https://pubs.acs.org/doi/10.1021/ci100050t) en 2010.
RDKit tiene una versión modificada que llaman *Morgan Fingerprint*. El parámetro más importante es el `radio`. Con un radio de 2, el fingerprint se denomina ECFP4, y con un radio de 3, ECFP6. Cuanto mayor sea el radio, más información sobre el entorno de cada átomo se incorpora al fingerprint.

La similitud entre dos fingerprints se calcula generalmente con la **Similitud de Tanimoto**. Esta métrica va de 0 (ninguna similitud) a 1 (moléculas idénticas). En la práctica, se considera que dos moléculas son similares si su similitud de Tanimoto es ≥ 0.8.

Primero calculamos el fingerprint para el sorafenib:

In [ ]:
from rdkit import DataStructs
fp_sorafenib = Chem.GetMorganFingerprint(sorafenib,radius=2)
fp_sorafenib

Este es el objeto que contiene el Fingerprint, si quieres ver sus valores tienes que usar una función diferente que se llama GetMorganFingerprintAsBitVector desde la clase AllChem

In [ ]:
from rdkit.Chem import AllChem

# Diccionario para almacenar cada Bit
bit_info = {}
# Crear el Fingerprint como una lista de 1 y 0
bit_vector = AllChem.GetMorganFingerprintAsBitVect(sorafenib, radius=2, nBits=2048, bitInfo=bit_info)
bit_list = [int(bit_vector[i]) for i in range(bit_vector.GetNumBits())]

# Imprimir
print("Primeros 50 bits:", bit_list[:50])
bits_encendidos = [i for i, b in enumerate(bit_list) if b == 1]
print(f"Bits encendidos: {len(bits_encendidos)} de 2048")
print(bits_encendidos)

Ahora con esa lista, vamos a imprimir las subestructuras que representa cada Bit

In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D
from IPython.display import display

# DrawMorganBit dibuja el entorno circular de un bit concreto
bits_encendidos = list(bit_info.keys())

img = Draw.DrawMorganBits(
    [(sorafenib, bit, bit_info) for bit in bits_encendidos],
    molsPerRow=8,
    subImgSize=(150, 150),
    legends=[f"Bit {b}" for b in bits_encendidos]
)
display(img)

El Morgan fingerprint no se almacena como un `np.array` regular. En los siguientes notebooks aprenderás cómo obtener también los vectores *normales* de los fingerprints.

Has calculado el fingerprint para el sorafenib, pero para calcular la similitud necesitas también los fingerprints de las otras moléculas. Escribe un `for-loop` para calcular el fingerprint de cada molécula en `mols_subset`:

In [ ]:
fp_mols = [Chem.GetMorganFingerprint( ___ ,radius = 2) for __ in ___ ]

<details>
<summary><strong>Solución:</strong></summary>

```python
    fp_mols = [Chem.GetMorganFingerprint( x ,radius = 2) for x in mols_subset]
```
</details>
<br>
Para calcular la similitud, usa la función descrita anteriormente `TanimotoSimilarity(fp1, fp2)`

In [ ]:
DataStructs.TanimotoSimilarity(fp_sorafenib,fp_mols[5])

Escribe un `for-loop` que calcule la similitud con el sorafenib para cada molécula en `fp_mols`.

In [ ]:
sorafenib_similarity = [DataStructs.TanimotoSimilarity(___ , ___ ) for x in ____]
sorafenib_similarity

<details>
<summary><strong>Solución:</strong></summary>

```python
    sorafenib_similarity=[DataStructs.TanimotoSimilarity(fp_sorafenib, x) for x in fp_mols]

```
</details>
<br>


In [ ]:
Draw.MolsToGridImage(mols_subset,legends = [str(x) for x in sorafenib_similarity],subImgSize=(300, 300))

Arriba puedes ver la similitud de cada molécula con el sorafenib. Una regla de oro comúnmente usada es que las moléculas con una similitud de 0.8 o mayor son suficientemente similares para considerarse una alternativa relevante. En nuestro caso, esto significa que solo probaríamos una molécula. La molécula con una similitud de 1.0 es idéntica al sorafenib — solo se representa de forma diferente.

## Ejercicio Práctico: Alternativas para el Antibiótico Norfloxacino

Como tarea algo más desafiante, ahora debes aplicar lo que has aprendido de forma independiente.
Básicamente, la tarea es muy similar a las anteriores, pero recibirás menos ayuda.
Primero, busca el `string` SMILES del **norfloxacino** (puedes buscarlo en PubChem o ChEMBL). A continuación, conviértelo en una molécula RDKit y muéstrala.

In [ ]:
# Esta es tu ultima tarea, usa esta celda
# para importar una libreria a la vez
from rdkit.Chem import AllChem as Chem
from rdkit.Chem import Draw
from rdkit.Chem.Descriptors import MolWt 
from rdkit.Chem.Crippen import MolLogP
from rdkit.Chem.Lipinski import NumHAcceptors, NumHDonors
from rdkit import DataStructs

In [ ]:
norfloxacin = "CCN1C=C(C(=O)C2=CC(=C(C=C21)N3CCNCC3)F)C(=O)O"
# Convierte el srting a molecula
norfloxacin = 
# Muestra la molecula
norfloxacin

A continuación, calcula los descriptores del norfloxacino que son importantes para la *Regla de los Cinco de Lipinski*.

In [ ]:
# Calcula el PM (peso molecular)
MW_norfloxacin = 

# Calcula el número de aceptores de puente de H
NumHAcceptors_norfloxacin = 

# Calcula el número de donadores de puente de H
NumHDonors_norfloxacin = 

# Calcula el logP
logP_norfloxacin = 

La siguiente celda muestra los descriptores calculados:

In [ ]:
print("MW:", MW_norfloxacin)
print("NumHAcceptors", NumHAcceptors_norfloxacin)
print("NumHDonors", NumHDonors_norfloxacin)
print("LogP",logP_norfloxacin)

En la siguiente celda se listan posibles alternativas al norfloxacino. Convierte los SMILES al formato `mol` y luego calcula los descriptores. La forma más sencilla es usar la notación `Lista = [función(x) for x in OtraLista]` utilizada varias veces anteriormente.

In [ ]:
# No olvides esta celda.
quinolones = ["C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O",
             "CN1CCN(CC1)C2=C(C=C3C(=C2F)N(C=C(C3=O)C(=O)O)CCF)F",
             "CCN1C=C(C(=O)C2=CC(=C(C(=C21)F)N3CCNC(C3)C)F)C(=O)O",
             "CC1CCC2=C3N1C=C(C(=O)C3=CC(=C2N4CCC(CC4)O)F)C(=O)O",
             "CC1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(=O)O"
             "CCN1C=C(C(=O)C2=CC(=C(C=C21)N3CCN(CC3)C)F)C(=O)O",
             "CN1CCN(CC1)C2=C(C=C3C4=C2SCCN4C=C(C3=O)C(=O)O)F",
             "CCN1C=C(C(=O)C2=CC(=C(N=C21)N3CCNCC3)F)C(=O)O",
             "CNC1CCCN(C1)C2=C(C=C3C(=C2OC)N(C=C(C3=O)C(=O)O)C4CC4)F",
             "CC1CN(CCN1)C2=C(C(=C3C(=C2)N(C=C(C3=O)C(=O)O)C4CC4)C)F",
             "C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(=O)O",
             "C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2C4(CC4)N)F)C(=O)O",
             "C[C@@H]1CN(C[C@@H](N1)C)C2=C(C(=C3C(=C2F)N(C=C(C3=O)C(=O)O)C4CC4)N)F",
             "CC1CN(CCN1)C2=C(C=C3C(=C2)N(C=C(C3=O)C(=O)O)C4=C(C=C(C=C4)F)F)F",
             "C1CN(CC1N)C2=C(C=C3C(=O)C(=CN(C3=N2)C4=C(C=C(C=C4)F)F)C(=O)O)F"]

In [ ]:
# Convierte los strings a moleculas
quinolones = 

In [ ]:
# Calcula los cuatro descriptores para todas las moléculas de la lista
MW_quinolones = 
NumHAcceptors_quinolones = 
NumHDonors_quinolones = 
logP_quinolones = 

En la siguiente celda puedes visualizar las moléculas con los descriptores calculados. No es necesario que entiendas el código en detalle.

Debes leer los valores tú mismo. (`Ctrl` + rueda del ratón para hacer zoom.)

In [ ]:
legend = []
for i in range(len(MW_quinolones)):
    legend.append("MW: "+str(round(MW_quinolones[i]))+"\n"+
                 "NumHAcceptors: "+str(NumHAcceptors_quinolones[i])+"\n"+
                 "NumHDonors: "+str(NumHDonors_quinolones[i])+"\n"+
                 "logP: "+str(round(logP_quinolones[i], 4)))

Draw.MolsToGridImage(quinolones, molsPerRow=3, legends = legend,
                    subImgSize=(250,150), useSVG=True)

Como prácticamente todas las moléculas siguen la *Regla de los Cinco de Lipinski*, en este punto no descartaremos ninguna molécula y en cambio calcularemos directamente la similitud con el norfloxacino. Para ello, primero hay que calcular los fingerprints y luego la similitud de Tanimoto.

In [ ]:
# Primero calcula los fingerprints de las quinolonas y el norfloxacino.
norfloxacin_fp = 
quinolones_fp = 

Ahora calcula las similitudes de `quinolones` con el norfloxacino.

En la celda siguiente puedes visualizar las similitudes.

In [ ]:
quinolones_similarity = 

In [ ]:
Draw.MolsToGridImage(quinolones, legends = [str(round(x, 2)) for x in quinolones_similarity],
                    subImgSize=(250,200), useSVG=True)

Como puedes ver, la mayoría de las moléculas no son particularmente similares al norfloxacino (al menos según la Similitud de Tanimoto). Sin embargo, cada una de estas moléculas es en realidad un antibiótico de amplio espectro que estuvo disponible, al menos en el pasado. Por lo tanto, no se puede confiar únicamente en la similitud entre moléculas. La actividad biológica depende de muchos factores — y la Similitud de Tanimoto captura solo una parte de ellos.

---

**Resumen de lo aprendido en este notebook:**
- Representar moléculas como objetos RDKit desde SMILES
- Manipular átomos y enlaces individuales con sus índices
- Calcular descriptores fisicoquímicos: LogP, PM, donadores/aceptores de H
- Aplicar la Regla de los Cinco de Lipinski para filtrar moléculas
- Calcular fingerprints de Morgan y similitud de Tanimoto
- Hacer búsquedas de similitud para encontrar alternativas moleculares

*NB-03 · Ciencia de Datos en Descubrimiento de Fármacos · UNAL 2026*